# VRC-3PO revision — R1c: shortcut-aware training (final detection experiment)

**Question:** if the network is *trained* so that recognising the environment earns it nothing, does it find a cybersickness signal that transfers to people it has never seen?

**Pre-registered** in `analysis/PREREGISTRATION_R1c.md` before this run. All four methods are reported whatever they show.

| method | what changes |
|---|---|
| `debias:select` | control: published CNN recipe, but model selection by *within-environment* validation AUC |
| `debias:balanced` | every (environment, class) cell gets equal training weight → environment carries no label information |
| `debias:adversarial` | gradient-reversal head removes environment information from the network's features |
| `debias:ranking` | trained only to rank elevated above non-elevated windows **from the same environment** |

Same 47 seated participants, same folds (repeats 0–2) and windows as the published-recipe ensemble, all 14 raw channels, 3-member ensembles.

**Decision rule:** a method supports a (restricted) detection claim only if its within-environment-and-minute AUC CI lies above 0.5 **and** its increment over environment + elapsed time has a CI above 0 — reported at 95% and at 98.75% (Bonferroni for 4 methods).

Runtime on a T4 ≈ 2.5–3.5 h, resumable (methods run most-informative first). Step 4 can be run at any time on finished folds.

In [ ]:
# ═══ 1 · Runtime ═══
!pip install -q keras-tcn
import tensorflow as tf, pandas as pd, numpy as np
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ═══ 2 · Drive + code (bundle v3) ═══
from google.colab import drive, files
from pathlib import Path
import zipfile, shutil, os
drive.mount('/content/drive')
REV = Path('/content/drive/MyDrive/vrc3po_revision')
BUNDLE = REV / 'vrc3po_revision_bundle_v3.zip'
if not BUNDLE.exists():
    print('Upload vrc3po_revision_bundle_v3.zip from the repo root on your Mac:')
    up = files.upload()
    shutil.copy([n for n in up if n.endswith('.zip')][0], BUNDLE)
root = Path('/content/vrc3po_code'); shutil.rmtree(root, ignore_errors=True); root.mkdir()
zipfile.ZipFile(BUNDLE).extractall(root); os.chdir(root)
assert (root / 'analysis/revision_debias.py').exists(), 'wrong bundle'
print((root / 'analysis/PREREGISTRATION_R1c.md').read_text())
!pip install -q pytest
!python -m pytest tests -q

In [ ]:
# ═══ 3 · Paths ═══
V11 = REV / 'vrc3po_benchmark_v1_1.csv'
OUT = REV / 'cv'          # same run store: the published-recipe ensemble and baselines are reused for pairing
assert V11.exists()
METHODS = ['debias:ranking', 'debias:balanced', 'debias:adversarial', 'debias:select']
print('finished published-recipe ensemble folds:', len(list((OUT / 'runs').glob('cnn_ensemble__fms_gt2__seated__cv__*.npz'))))

## Step 1 · Smoke test (1 fold, 2 epochs, ~2 min)

In [ ]:
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$REV/smoke_r1c" --models debias:ranking debias:adversarial --members 42 --repeats 1 --folds 5 --epochs 2
print('Smoke test passed.')

## Step 2 · Full run (GPU, ≈ 2.5–3.5 h, resumable)

In [ ]:
assert tf.config.list_physical_devices('GPU'), 'Switch to a GPU runtime.'
for m in METHODS:
    !python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --models {m} --members 42 123 456 --repeats 3 --folds 5

## Step 3 · Sensor-free baselines for these windows
Already computed for the full seated window set in the R1 run; this only fills anything missing (seconds).

In [ ]:
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --models nuisance:condition nuisance:condition_clock --repeats 5 --folds 5

## Step 4 · Summary and pre-registered decision → `results_revision_r1c.zip`

In [ ]:
import subprocess
proc = subprocess.Popen(['python', '-m', 'analysis.revision_cv', 'summarize', '--dataset', str(V11),
                         '--output-dir', str(OUT), '--bootstrap', '2000'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if line.startswith('['): print(line, end='')
assert proc.wait() == 0, 'summary failed'
S = OUT / 'summary'
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
p = pd.read_csv(S / 'primary_estimates.csv')
p = p[(p.scope == 'seated') & (p.label == 'fms_gt2') & (p.protocol == 'cv')]
e = pd.read_csv(S / 'environment_control.csv'); e = e[(e.scope == 'seated') & (e.label == 'fms_gt2')]
keep = ['cnn_ensemble', 'nuisance:condition', 'nuisance:condition_clock'] + METHODS
t = p[p.model.isin(keep)].merge(e, on=['label', 'scope', 'model'], how='left', suffixes=('', '_e'))
cols = ['model', 'repeats', 'auc_mean', 'within_condition_auc', 'within_condition_minute_auc',
        'within_condition_minute_ci_low', 'within_condition_minute_ci_high',
        'within_condition_minute_ci98_low', 'increment_over_env_clock',
        'increment_env_clock_ci_low', 'increment_env_clock_ci98_low']
display(t[cols].round(3))
print('\nPRE-REGISTERED DECISION')
for _, r in t[t.model.isin(METHODS)].iterrows():
    ok95 = r.within_condition_minute_ci_low > 0.5 and r.increment_env_clock_ci_low > 0
    ok98 = r.within_condition_minute_ci98_low > 0.5 and r.increment_env_clock_ci98_low > 0
    print(f"  {r.model:22s} 95%: {'SUPPORTS' if ok95 else 'no'}   98.75% (Bonferroni): {'SUPPORTS' if ok98 else 'no'}")
c = pd.read_csv(S / 'paired_comparisons.csv'); c = c[(c.scope == 'seated') & (c.label == 'fms_gt2') & c.model.isin(METHODS)]
print('\nPAIRED VS PUBLISHED-RECIPE ENSEMBLE (same folds)'); display(c.round(3))

bundle = Path('/content/results_revision_r1c'); shutil.rmtree(bundle, ignore_errors=True); bundle.mkdir()
shutil.copytree(S, bundle / 'cv_summary')
shutil.copy(root / 'analysis/PREREGISTRATION_R1c.md', bundle)
shutil.make_archive('/content/results_revision_r1c', 'zip', bundle)
shutil.copy('/content/results_revision_r1c.zip', REV / 'results_revision_r1c.zip')
os.chdir('/content'); drive.flush_and_unmount()
print('✓ MyDrive/vrc3po_revision/results_revision_r1c.zip — copy it into VRC-3PO/ on your Mac')